### MODELAGEM DA INFERENCIA DE CAUSALIDADE DE ACIDENTES DE TRANSITO

 - Usando ordered logit para modelar primeiramente (sem a causa pois ela possui endogeinedade) o que influencia probabilisticamente em um acidente de transito.
 - Para o segundo modelo e feito com a endogeinedade para saber o impacto que as variaveis causam no acidente de transito sem se preocupar com o efeito probabilistico de o quanto digamos 1 km a mais influencia da probabilidade de ser um acidente de alta letalidade. Somente para direcionar politica publica (o onde agir)

In [1]:
import polars as pl
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrix
import warnings

from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import StandardScaler
warnings.simplefilter('ignore', ConvergenceWarning)

import numpy as np

In [47]:
df = pl.read_parquet("./data/anuario_prf.parquet")
df = df.to_pandas()

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258737 entries, 0 to 258736
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   uf                      258737 non-null  category      
 1   br                      258737 non-null  category      
 2   km                      258737 non-null  int64         
 3   causa_acidente          258737 non-null  category      
 4   tipo_acidente           258737 non-null  category      
 5   fase_dia                258737 non-null  category      
 6   sentido_via             258737 non-null  category      
 7   condicao_metereologica  258737 non-null  category      
 8   tipo_pista              258737 non-null  category      
 9   tracado_via             258737 non-null  category      
 10  uso_solo                258737 non-null  category      
 11  pessoas                 258737 non-null  int64         
 12  veiculos                258737

In [49]:
df['fase_dia'].unique()

['Pleno dia', 'Amanhecer', 'Anoitecer', 'Plena Noite']
Categories (4, object): ['Pleno dia', 'Amanhecer', 'Anoitecer', 'Plena Noite']

In [50]:
fase_dia_ordem = ['Pleno dia', 'Amanhecer', 'Anoitecer', 'Plena Noite']

df['fase_dia'] = pd.Categorical(
    df['fase_dia'], 
    categories=fase_dia_ordem, 
    ordered=False 
)


tracado_ordem = ['Reta', 'Curva', 'Aclive_Declive', 'Intersecao', 'Especial', 'Outro']

# Aplique a ordem
df['tracado_via'] = pd.Categorical(
    df['tracado_via'], 
    categories=tracado_ordem, 
    ordered=False
)

uf_ordem = ['RS', 'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO']
df['uf'] = pd.Categorical(
    df['uf'], 
    categories=uf_ordem, 
    ordered=False
)


clima_ordem = ['Céu Claro', 'Nublado', 'Chuva', 'Garoa/Chuvisco', 'Sol', 'Ignorado', 'Nevoeiro/Neblina', 'Vento', 'Granizo', 'Neve']
df['condicao_metereologica'] = pd.Categorical(
    df['condicao_metereologica'], 
    categories=clima_ordem, 
    ordered=False
)

In [51]:
colunas_para_escalar = ['km', 'pessoas', 'veiculos']

scaler = StandardScaler()

df[colunas_para_escalar] = scaler.fit_transform(df[colunas_para_escalar])

In [52]:
crash_counts = df['br'].value_counts()

state_counts = df.groupby('br')['uf'].nunique()

/tmp/ipykernel_37435/2834029756.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  state_counts = df.groupby('br')['uf'].nunique()


In [53]:
br_stats = pd.DataFrame({
    'total_acidentes': crash_counts,
    'num_estados': state_counts
})

In [54]:
min_estados = 5
top_brs_stats = br_stats[
    br_stats['num_estados'] >= min_estados
].sort_values(by='total_acidentes', ascending=False)

In [55]:
TOP_BRS_LIST = top_brs_stats.head(5).index.tolist()

In [56]:
print(f"As Top 5 BRs (>= {min_estados} estados e mais acidentes) são:")
print(TOP_BRS_LIST)
print("\nEstatísticas completas das BRs filtradas:")
print(top_brs_stats)

As Top 5 BRs (>= 5 estados e mais acidentes) são:
['101', '116', '153', '163', '364']

Estatísticas completas das BRs filtradas:
     total_acidentes  num_estados
br                               
101            44646           11
116            40240           10
153             9856            8
163             8910            5
364             7961            5
230             5794            7
316             4450            5
20              2741            5
158             2634            8
110             1197            5
235             1144            5
226              996            5
0                669           27


In [57]:
for br in TOP_BRS_LIST:
    df[f'br_{br}'] = (df['br'] == br).astype(int)
    
    df[f'km_br_{br}'] = df['km'] * df[f'br_{br}']

In [58]:
risco_order = pd.CategoricalDtype(
    categories=['baixo', 'medio', 'alto'],
    ordered=True
)
df['risco_ordenado'] = df['risco'].astype(risco_order)

In [59]:
y = df['risco_ordenado'].cat.codes

In [60]:
main_effects = [f"br_{br}" for br in TOP_BRS_LIST]
interaction_effects = [f"km_br_{br}" for br in TOP_BRS_LIST]
all_new_terms = main_effects + interaction_effects
new = " + ".join(all_new_terms)

In [61]:
formula_modelo_1 = f"""
    km + {new} + pessoas + veiculos + C(uf) +
    C(fase_dia) + C(condicao_metereologica) +
    C(tipo_pista) + C(tracado_via) + C(uso_solo) +
    C(sentido_via) + C(tipo_acidente) + C(mes) + C(dia_semana_num)
"""

In [62]:
formula_modelo_1

'\n    km + br_101 + br_116 + br_153 + br_163 + br_364 + km_br_101 + km_br_116 + km_br_153 + km_br_163 + km_br_364 + pessoas + veiculos + C(uf) +\n    C(fase_dia) + C(condicao_metereologica) +\n    C(tipo_pista) + C(tracado_via) + C(uso_solo) +\n    C(sentido_via) + C(tipo_acidente) + C(mes) + C(dia_semana_num)\n'

In [63]:
X1 = dmatrix(formula_modelo_1, df, return_type='dataframe')
X1.drop(columns=['Intercept'], inplace=True)

In [64]:
model = OrderedModel(y, X1, distr='logit')

In [65]:
iteration_counter = [0]
DEBUG_MODE = False

def show_optimized_progress(params):
    """
    Callback otimizado que mostra o progresso.
    
    - No modo normal (DEBUG_MODE=False), recalcula o LL e o Gradiente
      apenas a cada 10 iterações para ser RÁPIDO.
    - No modo debug (DEBUG_MODE=True), recalcula a cada 1 iteração.
    """
    iteration_counter[0] += 1
    
    print_frequency = 1 if DEBUG_MODE else 25
    
    if (iteration_counter[0] % print_frequency == 0):
        
        neg_ll = -model.loglike(params)
        gradient = model.score(params)
        grad_norm = np.linalg.norm(gradient) 
        
        print(f"Iter: {iteration_counter[0]:<3} | Neg. LL: {neg_ll:<15.4f} | Grad. Norm: {grad_norm:<12.6f}")

In [66]:
print("Iniciando o ajuste do modelo (Modo Diagnóstico Lento)...")
print("="*60)
print(f"{'Iter':<3} | {'Neg. LL':<15} | {'Grad. Norm':<12}")
print("="*60)

fit_1 = model.fit(
    method='lbfgs', 
    callback=show_optimized_progress
)

print("="*60)
print("Ajuste concluído!")
print(fit_1.summary())

Iniciando o ajuste do modelo (Modo Diagnóstico Lento)...
Iter | Neg. LL         | Grad. Norm  
Iter: 25  | Neg. LL: 169313.9522     | Grad. Norm: 1487.019766 
Iter: 50  | Neg. LL: 169130.4375     | Grad. Norm: 213.871415  
Iter: 75  | Neg. LL: 169118.3505     | Grad. Norm: 56.832917   
Iter: 100 | Neg. LL: 169116.2252     | Grad. Norm: 90.591351   
Iter: 125 | Neg. LL: 169114.9861     | Grad. Norm: 28.847424   
Iter: 150 | Neg. LL: 169112.8375     | Grad. Norm: 34.965657   
Iter: 175 | Neg. LL: 169111.2723     | Grad. Norm: 36.446601   
Ajuste concluído!
                             OrderedModel Results                             
Dep. Variable:                      y   Log-Likelihood:            -1.6911e+05
Model:                   OrderedModel   AIC:                         3.384e+05
Method:            Maximum Likelihood   BIC:                         3.394e+05
Date:                Sat, 15 Nov 2025                                         
Time:                        21:48:25       

In [67]:
print(fit_1.summary())
fit_1.save("./results/inference1.pkl")

                             OrderedModel Results                             
Dep. Variable:                      y   Log-Likelihood:            -1.6911e+05
Model:                   OrderedModel   AIC:                         3.384e+05
Method:            Maximum Likelihood   BIC:                         3.394e+05
Date:                Sat, 15 Nov 2025                                         
Time:                        21:48:25                                         
No. Observations:              258737                                         
Df Residuals:                  258641                                         
Df Model:                          94                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
C(uf)[T.AC]                                        

In [72]:
formula_modelo_2 = f"""
    km + {new} + pessoas + veiculos + C(uf) +
    C(fase_dia) + C(condicao_metereologica) +
    C(tipo_pista) + C(tracado_via) + C(uso_solo) +
    C(sentido_via) + C(tipo_acidente) + C(mes) + C(dia_semana_num) +
    C(causa_acidente)
"""

In [74]:
X2 = dmatrix(formula_modelo_2, df, return_type='dataframe')
X2.drop(columns=['Intercept'], inplace=True)

In [75]:
model = OrderedModel(y, X2, distr='logit')

In [76]:
iteration_counter = [0]
print("Iniciando o ajuste do modelo (Modo Diagnóstico Lento)...")
print("="*60)
print(f"{'Iter':<3} | {'Neg. LL':<15} | {'Grad. Norm':<12}")
print("="*60)

fit_2 = model.fit(
    method='lbfgs', 
    callback=show_optimized_progress
)

print("="*60)
print("Ajuste concluído!")

Iniciando o ajuste do modelo (Modo Diagnóstico Lento)...
Iter | Neg. LL         | Grad. Norm  
Iter: 25  | Neg. LL: 168463.8684     | Grad. Norm: 538.212706  
Iter: 50  | Neg. LL: 168188.6296     | Grad. Norm: 281.738194  
Iter: 75  | Neg. LL: 168139.2400     | Grad. Norm: 147.977810  
Iter: 100 | Neg. LL: 168121.7645     | Grad. Norm: 230.023097  
Iter: 125 | Neg. LL: 168113.6298     | Grad. Norm: 89.924559   
Iter: 150 | Neg. LL: 168110.7315     | Grad. Norm: 31.906345   
Iter: 175 | Neg. LL: 168109.1435     | Grad. Norm: 59.732259   
Iter: 200 | Neg. LL: 168108.1172     | Grad. Norm: 27.627507   
Iter: 225 | Neg. LL: 168107.5977     | Grad. Norm: 49.151508   
Iter: 250 | Neg. LL: 168107.0992     | Grad. Norm: 14.837424   
Iter: 275 | Neg. LL: 168106.8935     | Grad. Norm: 16.580017   
Iter: 300 | Neg. LL: 168106.6563     | Grad. Norm: 16.518576   
Ajuste concluído!


In [77]:
print(fit_2.summary())
fit_2.save("./results/inference2.pkl")

                             OrderedModel Results                             
Dep. Variable:                      y   Log-Likelihood:            -1.6811e+05
Model:                   OrderedModel   AIC:                         3.366e+05
Method:            Maximum Likelihood   BIC:                         3.383e+05
Date:                Sat, 15 Nov 2025                                         
Time:                        23:24:07                                         
No. Observations:              258737                                         
Df Residuals:                  258566                                         
Df Model:                         169                                         
                                                                                                        coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------

In [131]:
df_sample = df.sample(n=65000, random_state=42)

In [132]:
df_sample

,uf,br,km,causa_acidente,tipo_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,...,km_br_101,br_116,km_br_116,br_153,km_br_153,br_163,km_br_163,br_364,km_br_364,risco_ordenado
58363,MG,365,-0.418749,Reação tardia ou ineficiente do condutor,Colisão traseira,Anoitecer,Crescente,Ignorado,Simples,Reta,...,-0.000000,0,-0.0,0,-0.000000,0,-0.0,0,-0.0,medio
129547,GO,153,0.502191,Reação tardia ou ineficiente do condutor,Colisão traseira,Pleno dia,Crescente,Céu Claro,Simples,Reta,...,0.000000,0,0.0,1,0.502191,0,0.0,0,0.0,baixo
186289,PR,369,-0.383497,Transtornos Mentais (exceto suicidio),Queda de ocupante de veículo,Pleno dia,Decrescente,Céu Claro,Simples,Reta,...,-0.000000,0,-0.0,0,-0.000000,0,-0.0,0,-0.0,baixo
152990,SC,282,1.008928,Ingestão de álcool pelo condutor,Colisão traseira,Plena Noite,Crescente,Céu Claro,Simples,Outro,...,0.000000,0,0.0,0,0.000000,0,0.0,0,0.0,baixo
10936,RJ,101,0.898768,Mal súbito do condutor,Colisão frontal,Pleno dia,Crescente,Nublado,Simples,Reta,...,0.898768,0,0.0,0,0.000000,0,0.0,0,0.0,baixo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119460,ES,101,0.167304,Trafegar com motocicleta (ou similar) entre as...,Queda de ocupante de veículo,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,...,0.167304,0,0.0,0,0.000000,0,0.0,0,0.0,baixo
164573,MG,40,0.793014,Velocidade Incompatível,Saída de leito carroçável,Pleno dia,Crescente,Céu Claro,Dupla,Reta,...,0.000000,0,0.0,0,0.000000,0,0.0,0,0.0,baixo
243553,SC,101,-0.938705,Trafegar com motocicleta (ou similar) entre as...,Colisão traseira,Pleno dia,Crescente,Céu Claro,Dupla,Reta,...,-0.938705,0,-0.0,0,-0.000000,0,-0.0,0,-0.0,baixo
15346,PR,373,-0.246899,Condutor Dormindo,Colisão lateral sentido oposto,Pleno dia,Crescente,Céu Claro,Simples,Reta,...,-0.000000,0,-0.0,0,-0.000000,0,-0.0,0,-0.0,baixo


In [133]:
formula_modelo_3 = f"""
    km + {new} + pessoas + veiculos + C(uf) +
    C(fase_dia) + C(condicao_metereologica) +
    C(tipo_pista) + C(tracado_via) + C(uso_solo) +
    C(tipo_acidente) + C(mes) + C(dia_semana_num) +
    C(causa_acidente) +
    C(causa_acidente) * C(fase_dia) +
    C(causa_acidente) * C(tipo_pista) +
    C(tipo_pista) * C(tracado_via)
    
"""
#   C(sentido_via)

In [134]:
X3 = dmatrix(formula_modelo_3, df_sample, return_type='dataframe')
X3.drop(columns=['Intercept'], inplace=True)

In [135]:
X3

,C(uf)[T.AC],C(uf)[T.AL],C(uf)[T.AP],C(uf)[T.AM],C(uf)[T.BA],C(uf)[T.CE],C(uf)[T.DF],C(uf)[T.ES],C(uf)[T.GO],C(uf)[T.MA],...,br_153,br_163,br_364,km_br_101,km_br_116,km_br_153,km_br_163,km_br_364,pessoas,veiculos
58363,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.000000,-0.0,-0.000000,-0.0,-0.0,0.184851,0.007886
129547,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.000000,0.0,0.502191,0.0,0.0,0.636918,0.889690
186289,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.000000,-0.0,-0.000000,-0.0,-0.0,-0.267216,-0.873917
152990,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.184851,0.007886
10936,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.898768,0.0,0.000000,0.0,0.0,0.184851,0.889690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119460,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.167304,0.0,0.000000,0.0,0.0,0.184851,0.007886
164573,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,1.088985,-0.873917
243553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.938705,-0.0,-0.000000,-0.0,-0.0,0.184851,0.007886
15346,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.000000,-0.0,-0.000000,-0.0,-0.0,0.184851,1.771494


In [136]:
y_sample_aligned = y[X3.index]

In [137]:
iteration_counter = [0]
model = OrderedModel(y_sample_aligned, X3, distr='logit')

In [ ]:
DEBUG_MODE = False
iteration_counter = [0]
print("Iniciando o ajuste do modelo (Modo Diagnóstico Lento)...")
print("="*60)
print(f"{'Iter':<3} | {'Neg. LL':<15} | {'Grad. Norm':<12}")
print("="*60)

fit_3 = model.fit(
        method='lbfgs', 
        callback=show_optimized_progress,
        maxiter=1000 # Dando um pouco mais de "corda" para o modelo complexo
    )

print("="*60)
print("Ajuste concluído!")



Iniciando o ajuste do modelo (Modo Diagnóstico Lento)...
Iter | Neg. LL         | Grad. Norm  
Iter: 25  | Neg. LL: 42092.1813      | Grad. Norm: 188.717649  
Iter: 50  | Neg. LL: 41981.3514      | Grad. Norm: 114.740926  
Iter: 75  | Neg. LL: 41940.7582      | Grad. Norm: 90.042553   
